# Bulk download files from Vocareum S3 bucket to Databricks Free Edition volume leveraging boto3

## Set all necessary widget values before running cell 3
- s3_bucket_name e.g. 's3-dle-ad53440a6...'
- s3_bucket_path e.g. 'project_export/json/'
- volume_path e.g. '/Volumes/workspace/default/boto3-s3-demo'.  Note that the volume must be created before running this download.

In [0]:
dbutils.widgets.text("s3_bucket_name", "")
dbutils.widgets.text("s3_bucket_path", "")
dbutils.widgets.text("volume_path", "")

In [0]:
bucket_name = dbutils.widgets.get("s3_bucket_name")
s3_path = dbutils.widgets.get("s3_bucket_path")
volume_path = dbutils.widgets.get("volume_path")

In [0]:
scope_name = "aws-vocareum-secret-scope"

## Initialize s3 client using boto3 and AWS user credentials in Vocareum

In [0]:
import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id=dbutils.secrets.get(scope=scope_name, key="aws_access_key_id"),
    aws_secret_access_key=dbutils.secrets.get(scope=scope_name, key="aws_secret_access_key"),
    aws_session_token=dbutils.secrets.get(scope=scope_name, key="session_token")
)

## Instantiate page iterator to list all objects

In [0]:
paginator = s3.get_paginator('list_objects_v2')

page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=s3_path)

## Download files from S3 to volume
Some possible enhancements to this cell: add logging, show download progress, or parallelize the operation.

In [0]:
for page in page_iterator:
  if 'Contents' in page:
    for obj in page['Contents']:
      response = s3.get_object(Bucket=bucket_name, Key=obj['Key'])
      with open(f"{volume_path}/{obj['Key'].split('/')[-1]}", 'wb') as f:
        f.write(response['Body'].read())
  else:
    print(f"No objects found in s3://{bucket_name}/{s3_path}")